# 🎓 Diya LoRA Training — Google Colab

Kaggle ki GPU quota **sirf photo banane** ke liye bachao.
Training yahan Colab pe chalegi — **free, alag quota**.

### Kaise chalayein
1. Upar **Runtime → Change runtime type → T4 GPU** chuno
2. Left sidebar me 🔑 **Secrets** kholo, 4 secret daalo (neeche list hai)
3. **Runtime → Run all**
4. Laptop band mat karo. Telegram pe progress aati rahegi.

Khatam hone pe LoRA **apne aap Kaggle pe upload** ho jaayegi,
aur server use utha lega. Kuch download nahi karna.


## 1. Secrets

Left sidebar → 🔑 **Secrets** → **Add new secret**.
Har ek ka **Notebook access** toggle ON karna zaroori hai.

| Name | Value |
|---|---|
| `KAGGLE_USERNAME` | `ukboy7u787` |
| `KAGGLE_KEY` | tumhara `KGAT_...` token |
| `TELEGRAM_BOT_TOKEN` | bot token |
| `TELEGRAM_CHAT_ID` | `6486771356` |


In [ ]:
import os, sys, subprocess, time
from google.colab import userdata

NEED = ["KAGGLE_USERNAME", "KAGGLE_KEY", "TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID"]
miss = []
for k in NEED:
    try:
        v = userdata.get(k)
        if not v:
            raise ValueError
        os.environ[k] = v
    except Exception:
        miss.append(k)

if miss:
    raise SystemExit(f"❌ Ye secrets daalo (sidebar 🔑): {', '.join(miss)}")
os.environ["KAGGLE_API_TOKEN"] = os.environ["KAGGLE_KEY"]
print("✅ secrets mil gaye")

g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
if g.returncode:
    raise SystemExit("❌ GPU nahi mila. Runtime → Change runtime type → T4 GPU")
print("✅ GPU:", g.stdout.strip())

## 2. Packages (~3 min)

In [ ]:
%pip install -q "diffusers>=0.31" "transformers>=4.44" accelerate safetensors \
    peft bitsandbytes kaggle requests pillow
print("✅ installed")

## 3. Training code + dataset

Code GitHub se, dataset Kaggle se — dono apne aap.


In [ ]:
!rm -rf /content/aspto /content/data && mkdir -p /content/data
!git clone -q --depth 1 https://github.com/Deepanshu-8126/aspto.git /content/aspto
assert os.path.exists("/content/aspto/training/kaggle_lora_train.py"), "train script nahi mila"
print("✅ code aa gaya")

!kaggle datasets download -q -d ukboy7u787/diya-lora-dataset -p /content/data --unzip
imgs = [f for f in os.listdir("/content/data") if f.lower().endswith((".png", ".jpg", ".jpeg"))]
caps = [f for f in os.listdir("/content/data") if f.endswith(".txt")]
print(f"✅ dataset: {len(imgs)} images, {len(caps)} captions")
assert len(imgs) >= 5, "dataset khaali hai — Kaggle creds check karo"

## 4. Training shuru

~2 ghante (2000 steps). Har **400 steps** pe Telegram pe preview aayegi.
Har **150 steps** pe save hota hai — beech me disconnect ho jaaye to bhi
jo ban chuka hai wo bach jaata hai.


In [ ]:
STEPS = 2000          # identity curve 900 pe flat nahi hui thi, isliye 2000

!cd /content/aspto/training && python kaggle_lora_train.py \
    --data /content/data \
    --out /content/diyarai_sdxl_lora.safetensors \
    --steps {STEPS} --save_every 150 --preview_every 400

LORA = "/content/diyarai_sdxl_lora.safetensors"
assert os.path.exists(LORA), "❌ LoRA nahi bani — upar ka error dekho"
print(f"✅ LoRA ready: {os.path.getsize(LORA)/1e6:.0f} MB")

## 5. Kaggle pe wapas upload

Server isi dataset se LoRA uthata hai. Upload hote hi next start pe
nayi LoRA use hogi — kuch aur nahi karna.


In [ ]:
import json, shutil, requests

up = "/content/upload"
shutil.rmtree(up, ignore_errors=True); os.makedirs(up)
shutil.copy(LORA, f"{up}/diyarai_sdxl_lora.safetensors")
json.dump({"id": "ukboy7u787/diyarai-sdxl-lora", "title": "diyarai-sdxl-lora"},
          open(f"{up}/dataset-metadata.json", "w"))

!cd {up} && kaggle datasets version -m "Colab: {STEPS} steps, 12 saaf images" 

requests.post(
    f"https://api.telegram.org/bot{os.environ['TELEGRAM_BOT_TOKEN']}/sendMessage",
    data={"chat_id": os.environ["TELEGRAM_CHAT_ID"],
          "text": f"✅ Colab training poori — {STEPS} steps\n"
                  "LoRA Kaggle pe chadh gayi.\n\n"
                  "Ab Telegram pe /night bhejo, server nayi LoRA utha lega."},
    timeout=20)
print("✅ ho gaya — Telegram pe /night bhejo")

---

## 🧯 Dikkat aaye to

| problem | hal |
|---|---|
| `GPU nahi mila` | Runtime → Change runtime type → **T4 GPU** |
| `secrets daalo` | sidebar 🔑 me daalo + **Notebook access** ON |
| `dataset khaali` | `KAGGLE_KEY` galat hai |
| beech me disconnect | dobara Run all — 150-step checkpoints bach jaate hain |
| `CUDA out of memory` | Runtime → Restart, phir Run all |

## 🔋 Quota ka hisaab

| | pehle | ab |
|---|---|---|
| Kaggle | training + server dono | **sirf server** |
| Colab | kuch nahi | **training** |

Kaggle ki poori 30h ab photo banane me jaayegi.
